In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

# CONFIG

# Numeric scenario label (0 = normal, >0 = suspicious scenario)
SCENARIO_COL = "Scenario_Main"

# Folder with OOF tables from binary OOF script
OOF_ROOT = Path("./out_oof_meta")

VARIANTS: List[str] = ["baseline", "rose"]
RANDOM_STATE = 42


# HELPERS

def list_waves_from_oof(variant: str) -> List[str]:
    """List waves for a variant based on folders."""
    base = OOF_ROOT / variant
    if not base.exists():
        return []
    return sorted([p.name for p in base.iterdir() if p.is_dir()])


def load_oof_tables(variant: str, wave: str):
    """Load train/test OOF tables for a wave."""
    base = OOF_ROOT / variant / wave
    tr_path = base / "train_with_oof.csv"
    te_path = base / "test_with_pred.csv"
    if not tr_path.exists() or not te_path.exists():
        raise FileNotFoundError(
            f"Missing OOF files for {variant}/{wave}: {tr_path} / {te_path}"
        )
    tr = pd.read_csv(tr_path)
    te = pd.read_csv(te_path)
    return tr, te


# MULTI-CLASS META MODEL

def run_multiclass_meta():
    """Train and evaluate multi-class model on OOF scores."""
    rows = []

    for variant in VARIANTS:
        waves = list_waves_from_oof(variant)
        if not waves:
            print(f"[skip] No waves found for variant={variant}")
            continue

        print(f"\n=== VARIANT: {variant} ===")
        for wave in waves:
            try:
                tr, te = load_oof_tables(variant, wave)
            except FileNotFoundError as e:
                print(f"  [skip {variant}/{wave}] {e}")
                continue

            if SCENARIO_COL not in tr.columns or SCENARIO_COL not in te.columns:
                print(f"  [skip {variant}/{wave}] no '{SCENARIO_COL}' in tables")
                continue

            # scenario labels (numeric)
            y_tr = tr[SCENARIO_COL].fillna(0).astype(int).to_numpy()
            y_te = te[SCENARIO_COL].fillna(0).astype(int).to_numpy()

            # only suspicious transactions (scenario > 0)
            mask_tr = y_tr > 0
            mask_te = y_te > 0

            if mask_tr.sum() == 0 or mask_te.sum() == 0:
                print(f"  [skip {variant}/{wave}] no suspicious rows for multi-class")
                continue

            # OOF-based features from binary models
            oof_cols = [c for c in tr.columns if c.startswith("oof_") and c.endswith("_bin")]
            if not oof_cols:
                print(f"  [skip {variant}/{wave}] no OOF columns (oof_*_bin)")
                continue

            pred_cols = [c.replace("oof_", "pred_") for c in oof_cols]
            missing_pred = [c for c in pred_cols if c not in te.columns]
            if missing_pred:
                print(f"  [skip {variant}/{wave}] missing test preds: {missing_pred}")
                continue

            X_tr = tr.loc[mask_tr, oof_cols].to_numpy()
            X_te = te.loc[mask_te, pred_cols].to_numpy()
            y_tr_s = y_tr[mask_tr]
            y_te_s = y_te[mask_te]

            print(
                f"  [WAVE] {wave}: "
                f"suspicious train={X_tr.shape[0]}, suspicious test={X_te.shape[0]}"
            )

            # multi-class classifier
            clf = LogisticRegression(
                multi_class="multinomial",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )
            clf.fit(X_tr, y_tr_s)
            y_pred = clf.predict(X_te)

            acc = accuracy_score(y_te_s, y_pred)
            f1_macro = f1_score(y_te_s, y_pred, average="macro")
            prec_macro = precision_score(
                y_te_s, y_pred, average="macro", zero_division=0
            )
            rec_macro = recall_score(
                y_te_s, y_pred, average="macro", zero_division=0
            )

            print(
                f"    [MULTI-CLASS TEST] acc={acc:.4f}, F1_macro={f1_macro:.4f}, "
                f"precision_macro={prec_macro:.4f}, recall_macro={rec_macro:.4f}"
            )

            rows.append(
                {
                    "variant": variant,
                    "wave": wave,
                    "n_train_suspicious": int(mask_tr.sum()),
                    "n_test_suspicious": int(mask_te.sum()),
                    "accuracy": float(acc),
                    "f1_macro": float(f1_macro),
                    "precision_macro": float(prec_macro),
                    "recall_macro": float(rec_macro),
                }
            )

            # per-class reports for this wave
            out_dir = OOF_ROOT / variant / wave
            out_dir.mkdir(parents=True, exist_ok=True)

            # confusion matrix
            cm = confusion_matrix(y_te_s, y_pred)
            classes = np.unique(y_te_s)
            cm_df = pd.DataFrame(cm, index=classes, columns=classes)
            cm_df.index.name = "true"
            cm_df.columns.name = "pred"
            cm_df.to_csv(out_dir / "scenario_confusion_matrix.csv")

            # per-class metrics
            rep = classification_report(
                y_te_s,
                y_pred,
                output_dict=True,
                zero_division=0,
            )
            rep_df = pd.DataFrame(rep).transpose()
            rep_df.to_csv(out_dir / "scenario_classification_report.csv")

            # save test with predictions
            te_out = te.copy()
            te_out["Scenario_Pred"] = -1
            te_out.loc[mask_te, "Scenario_Pred"] = y_pred
            te_out.to_csv(out_dir / "test_with_scenario_pred.csv", index=False)

    if not rows:
        print("No rows produced for multi-class meta experiment.")
        return

    res = pd.DataFrame(rows)
    out_path = OOF_ROOT / "multiclass_meta_summary.csv"
    res.to_csv(out_path, index=False)
    print(f"\nSaved multi-class summary to: {out_path}")


if __name__ == "__main__":
    run_multiclass_meta()